<!-- nav -->
[← 10 · Answers you can trust](10_Answers_You_Can_Trust.ipynb) · [Home](../../README.md) · [12 · Deciding what you can now →](12_Deciding_What_You_Can_Now.ipynb)

# 11 · Constraints over collections

**The problem.** The latest reading isn't the whole story. A logger records the temperature every few minutes, and a
shipment that spent an hour at 12 °C may be back at 5 °C by the time it's checked. The real constraints are about *all*
the readings: every one in range, or at most two brief excursions, never above 12 °C. And about the loggers a shipment
carries: each one's battery must last the trip.

This case study writes constraints over lists and over an object's related objects, with **quantifiers**.

In [1]:
import { Simplifiers, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import { Terms } from "@mbse/expressions/Framework";
import { cold_chain, show } from "./toolkit.js";

const store = cold_chain();
const [Expression, Term] = [store.Expressions.Framework.Expression, store.Expressions.Framework.Term];
const shipments: any[] = store.transaction(() => [
  store.Shipment((s: any) => s.id("S1").readings([4.0, 5.5, 6.1, 5.0])),
  store.Shipment((s: any) => s.id("S2").readings([4.0, 8.6, 9.1, 5.0])),
  store.Shipment((s: any) => s.id("S3").readings([4.0, 12.5, 14.0, 13.2])),
  store.Shipment((s: any) => s.id("S4"))]); // no readings uploaded
const [steady, blip, cooked, silent] = shipments;
const [shipment, c] = [E.variable("shipment"), E.variable("c")];
const table = (constraint: any) =>
  Object.fromEntries(shipments.map((s: any) => [s.id.get(), Simplifiers.OfAny(constraint, { shipment: s }).value]));

<a id="T11.1"></a>
## 11.1 · A list is a collection

`readings` is a list property (a container, in mbse-schemas' terms). Reading it gives a **collection**, which a handful
of operations take: `count`, `item` (by position, from 0), `in_`, `sum`, `min`, `max` and `unique`.

These quick questions aren't constraints to keep, so each is written as a term alone, `Term(writer)`: a value with no
`Expression` object around it, which evaluates the same way.

In [2]:
const questions: any[] = store.transaction(() => [shipment.readings, shipment.readings.count(), shipment.readings.item(0n),
  shipment.readings.max(), E.literal(9.1).in_(shipment.readings), shipment.readings.unique()].map((writer) => Term(writer)));
for (const expression of questions) {
  console.log(show(expression).padEnd(40), String(Simplifiers.OfAny(expression, { shipment: blip }).value));
}

get(shipment, 'readings')                [4.0, 8.6, 9.1, 5.0]


count(get(shipment, 'readings'))         4


item(get(shipment, 'readings'), 0)       4


max(get(shipment, 'readings'))           9.1


in(9.1, get(shipment, 'readings'))       true


unique(get(shipment, 'readings'))        true


Collections treat unknowns as everything else does. There's no item at position 9, so it's unknown,
not an error; the silent shipment has no readings at all, so everything about them is unknown:

In [3]:
const more: any[] = store.transaction(() =>
  [shipment.readings.item(9n), shipment.readings.count(), shipment.readings.max()].map((writer) => Term(writer)));
for (const expression of more) {
  console.log(show(expression).padEnd(40), String(Simplifiers.OfAny(expression, { shipment: blip }).value), String(Simplifiers.OfAny(expression, { shipment: silent }).value));
}

item(get(shipment, 'readings'), 9)       null null


count(get(shipment, 'readings'))         4 null


max(get(shipment, 'readings'))           9.1 null


<a id="T11.2"></a>
## 11.2 · Quantifiers

A **quantifier** binds a name to each item in turn, here `c`, and tests a body. `all` holds when the body holds for
every item, `any` when it holds for one, and `count_where` counts the items it holds for:

In [4]:
const every_in_range = store.transaction(() =>
  Expression((e: any) => e.root(shipment.readings.all("c", E.literal(2.0).le(c).and_(c.le(8.0))))));
console.log(show(every_in_range));
console.log(table(every_in_range));

all(get(shipment, 'readings'), c => and(le(2.0, c), le(c, 8.0)))


{ S1: true, S2: false, S3: false, S4: null }


The steady shipment passes and the others fail; the silent one is unknown. The quantifier is a term of its own
kind, which binds `c` the way a let does, so `c` means nothing outside the body.

The real constraint allows brief excursions: at most two readings above 8 °C, and none above 12 °C. That's a count and a
maximum:

In [5]:
const excursions = shipment.readings.count_where("c", c.gt(8.0)); // a writer, not yet written
const allowed = store.transaction(() => Expression((e: any) => e.root(excursions.le(2n).and_(shipment.readings.max().le(12.0)))));
console.log(show(allowed));
console.log(table(allowed));

and(le(count(get(shipment, 'readings'), c => gt(c, 8.0)), 2), le(max(get(shipment, 'readings')), 12.0))


{ S1: true, S2: true, S3: false, S4: null }


The blip is allowed now; the cooked shipment still isn't.

<a id="T11.3"></a>
## 11.3 · Empty collections

An empty collection is data too, and quantifiers answer the way logic does: every item of nothing satisfies anything
(`all` is true), none satisfies something (`any` is false), and the count is 0. What has no answer is the greatest of
nothing, so `max` is unknown, and a constraint that relies on it says so:

In [6]:
const [empty, cases] = store.transaction(() => [store.Shipment((s: any) => s.id("S5").readings([])), [
  ["every_in_range", every_in_range], ["any above 8", Term(shipment.readings.any("c", c.gt(8.0)))],
  ["excursions", Term(excursions)], ["max", Term(shipment.readings.max())], ["allowed", allowed]]]);
for (const [name, constraint] of cases as [string, any][]) {
  console.log(name.padEnd(16), String(Simplifiers.OfAny(constraint, { shipment: empty }).value));
}

every_in_range   true


any above 8      false


excursions       0


max              null


allowed          null


`allowed` for an empty list is unknown, not true: `excursions` is 0, which is fine, but the maximum is unknown. A
shipment that logged nothing hasn't shown it stayed cold, and the constraint doesn't pretend otherwise. If an empty log
*should* pass, say so explicitly, with `count`:

In [7]:
// A writer builds on a writer or a term value, such as an expression's root.
const lenient = store.transaction(() => Expression((e: any) => e.root(shipment.readings.count().eq(0n).or_(Terms.term(allowed)))));
console.log(Simplifiers.OfAny(lenient, { shipment: empty }).value, Simplifiers.OfAny(lenient, { shipment: cooked }).value);

true false


<a id="T11.4"></a>
## 11.4 · An object's related objects

A shipment carries loggers, through the relation `Carries`, whose entries record the slot each logger rides in: a
shipment's adjacency `loggers`. `entries(name)` gives an object's entries in an adjacency as a collection of
**records**, which `get` reads like objects: the linked logger, and the entry's own properties.

In [8]:
const [trip, entries_] = store.transaction(() => {
  const l17 = store.Logger((l: any) => l.serial("L-17").battery(80n));
  const l22 = store.Logger((l: any) => l.serial("L-22").battery(15n));
  const trip = store.Shipment((s: any) => s.id("S6").readings([5.0])
    .loggers([(e: any) => e.slot(1n).logger(l17), (e: any) => e.slot(2n).logger(l22)]));
  return [trip, [shipment.entries("loggers").count(), shipment.entries("loggers").item(1n).get("slot"),
    shipment.entries("loggers").item(1n).get("logger").get("serial")].map((writer) => Term(writer))];
});
const e = E.variable("e");
for (const expression of entries_) {
  console.log(show(expression).padEnd(70), String(Simplifiers.OfAny(expression, { shipment: trip }).value));
}

count(entries(shipment, 'loggers'))                                    2


get(item(entries(shipment, 'loggers'), 1), 'slot')                     2


get(get(item(entries(shipment, 'loggers'), 1), 'logger'), 'serial')    L-22


`.get("logger")` is the writer of `get(e, 'logger')`; `e.logger` would be the same. So "every logger's battery
lasts the trip" is a quantifier over the entries:

In [9]:
const [charged, flat] = store.transaction(() => [
  Expression((x: any) => x.root(shipment.entries("loggers").all("e", e.logger.battery.ge(20n)))),
  Term(shipment.entries("loggers").count_where("e", e.logger.battery.lt(20n)))]);
console.log(show(charged));
console.log(Simplifiers.OfAny(charged, { shipment: trip }).value);
console.log(String(Simplifiers.OfAny(flat, { shipment: trip }).value));

all(entries(shipment, 'loggers'), e => ge(get(get(e, 'logger'), 'battery'), 20))


false


1


<a id="T11.5"></a>
## 11.5 · Building a list or a mapping

A collection needn't come from the data. The **container constructors** build one from terms: `E.list_(a, b, ...)` a
list of its arguments, in order, and `E.mapping([key, value], ...)` a mapping of keys to values, which `item` reads by
key. Each item is a term, evaluated as any other, so a list can gather values from several properties:

In [10]:
const [late, built] = store.transaction(() => [store.Shipment((s: any) => s.id("S7").celsius(5.0).readings([9.5, 6.0])),
  [E.list_(shipment.celsius, shipment.readings.item(0n)), E.mapping(["lo", 2.0], ["hi", 8.0]),
    E.mapping(["lo", 2.0], ["hi", 8.0]).item("hi")].map((writer) => Term(writer))]);
for (const expression of built as any[]) {
  console.log(show(expression).padEnd(62), String(Simplifiers.OfAny(expression, { shipment: late }).value));
}

[get(shipment, 'celsius'), item(get(shipment, 'readings'), 0)] [5.0, 9.5]


{'lo': 2.0, 'hi': 8.0}                                         {'lo': 2.0, 'hi': 8.0}


item({'lo': 2.0, 'hi': 8.0}, 'hi')                             8


This shipment's latest reading is fine, but its first was 9.5 °C. "The first and the latest readings are both in
range", with the range written once, as a mapping a let names, is a quantifier over a list:

In [11]:
const limits = E.variable("limits");
const ends_in_range = store.transaction(() => Expression((e: any) => e.root(E.let_("limits", E.mapping(["lo", 2.0], ["hi", 8.0]),
  E.list_(shipment.celsius, shipment.readings.item(0n)).all("c", limits.item("lo").le(c).and_(c.le(limits.item("hi"))))))));
console.log(show(ends_in_range));
console.log(String(Simplifiers.OfAny(ends_in_range, { shipment: late }).value), String(Simplifiers.OfAny(ends_in_range, { shipment: silent }).value));

let limits = {'lo': 2.0, 'hi': 8.0} in all([get(shipment, 'celsius'), item(get(shipment, 'readings'), 0)], c => and(le(item(limits, 'lo'), c), le(c, item(limits, 'hi'))))


false null


It fails on the first reading, and for the silent shipment, which reported neither, it is unknown. A constructed
collection is a term like any other: it is stored, validated and evaluated with the constraint that holds it.

## Why it works this way

- **Quantifiers instead of loops.** A quantifier visits each item of data once, so a constraint with quantifiers is
  still bounded, total and pure.
- **Collections come from data, or from terms.** A literal is one basic value, so a collection is a list property, an
  object's entries, a variable the caller binds, or one the container constructors `list` and `mapping` build from
  terms.
- **The empty case is honest.** `all` of nothing is true because nothing violates it; the maximum of nothing doesn't
  exist. Constraints that need an answer for the empty case say what it is.

Collections and quantifiers, in full:
[CAP-096: collections in expressions](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-096.md#collections).

<!-- nav -->
[← 10 · Answers you can trust](10_Answers_You_Can_Trust.ipynb) · [Home](../../README.md) · [12 · Deciding what you can now →](12_Deciding_What_You_Can_Now.ipynb)